# 23. Semantic Bridge Accord-only v2

v1의 동일한 12개 `bridge_phrase`를 기존 Accord 92개 중 1~2개 또는 `ABSTAIN`으로만
변환한다. 이 실험은 schema 안정성과 Accord mapping 후보만 확인하며 Retrieval, 사람 평가,
Canonical Note, 자연어 전처리를 수행하지 않는다.

v1 Notebook/checkpoint/사람 평가 파일은 읽기만 하며 수정하지 않는다.


In [1]:
import hashlib
import json
import os
import pathlib
import time
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
from IPython.display import Markdown, display

ROOT = pathlib.Path.cwd()
OUTPUT_DIR = ROOT / "analysis_outputs"
V1_NOTEBOOK_PATH = ROOT / "22_semantic_bridge_pilot.ipynb"
V1_CHECKPOINT_PATH = OUTPUT_DIR / "22_semantic_bridge_pilot_checkpoint.json"
ACCORD_DICTIONARY_PATH = OUTPUT_DIR / "10_accord_dictionary.csv"
V2_CHECKPOINT_PATH = OUTPUT_DIR / "23_semantic_bridge_accord_only_v2_checkpoint.json"
COMPARISON_PATH = OUTPUT_DIR / "23_semantic_bridge_accord_only_v2_comparison.csv"

required_paths = [V1_NOTEBOOK_PATH, V1_CHECKPOINT_PATH, ACCORD_DICTIONARY_PATH]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")

PILOT_VERSION = "semantic-bridge-accord-only-v2"
MODEL_ID = "gpt-5.4-nano"
ENDPOINT = "https://gms.ssafy.io/gmsapi/api.openai.com/v1/chat/completions"
TEMPERATURE = 0
MAX_RETRIES = 2
REQUEST_TIMEOUT_SECONDS = 90


def file_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_hash(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def write_json(path, value):
    path = pathlib.Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)


v1_input_hashes = {
    "22_notebook_sha256": file_sha256(V1_NOTEBOOK_PATH),
    "22_checkpoint_sha256": file_sha256(V1_CHECKPOINT_PATH),
}
print(f"Pilot: {PILOT_VERSION}")
print(f"v1 Notebook SHA256: {v1_input_hashes['22_notebook_sha256']}")
print(f"v1 Checkpoint SHA256: {v1_input_hashes['22_checkpoint_sha256']}")


Pilot: semantic-bridge-accord-only-v2
v1 Notebook SHA256: 4a72e833b44f1c10c801d638e429655932c0965400c0882b263a1b4d1cbb1aa6
v1 Checkpoint SHA256: 7af3984421d33690b8cf51e7f5e50db5f2667bf9d7d4f7efcf83bb9db879092a


## 1. v1의 동일 입력과 Accord 92개 동결

Query와 `bridge_phrase`는 v1 checkpoint의 selection hash에 연결된 12개를 그대로 사용한다.
Accord는 `10_accord_dictionary.csv`에서 읽고 정렬한 exact name 92개뿐이다.


In [2]:
v1_checkpoint = json.loads(V1_CHECKPOINT_PATH.read_text(encoding="utf-8"))
v1_preregistration = json.loads(
    (OUTPUT_DIR / "22_semantic_bridge_pilot_preregistration.json").read_text(encoding="utf-8")
)

selected_queries = v1_preregistration["selection"]["selected_queries"]
query_inputs = [
    {
        "query_id": row["query_id"],
        "query_text": row["query_text"],
        "resolution_type": row["resolution_type"],
        "bridge_phrase": row["bridge_phrase"],
    }
    for row in selected_queries
]
v1_result_by_id = {row["query_id"]: row for row in v1_checkpoint["llm_results"]}

if len(query_inputs) != 12 or len({row["query_id"] for row in query_inputs}) != 12:
    raise RuntimeError("v1 고정 Query가 unique 12개가 아닙니다.")
if set(v1_result_by_id) != {row["query_id"] for row in query_inputs}:
    raise RuntimeError("v1 selection과 v1 LLM result ID가 다릅니다.")
if v1_checkpoint["selection_annotation_sha256"] != v1_preregistration["selection"]["selection_annotation_sha256"]:
    raise RuntimeError("v1 selection annotation hash가 다릅니다.")

accord_dictionary_df = pd.read_csv(ACCORD_DICTIONARY_PATH)
allowed_accords = sorted(accord_dictionary_df["accord"].dropna().astype(str).unique())
if len(allowed_accords) != 92:
    raise RuntimeError(f"Accord vocabulary가 92개가 아닙니다: {len(allowed_accords)}")
allowed_accord_set = set(allowed_accords)
accord_vocabulary_hash = json_hash(allowed_accords)
query_input_hash = json_hash([
    {"query_id": row["query_id"], "bridge_phrase": row["bridge_phrase"]}
    for row in query_inputs
])

v1_schema_valid_count = sum(not row["validation_error"] for row in v1_checkpoint["llm_results"])
v1_validation_failure_count = sum(bool(row["validation_error"]) for row in v1_checkpoint["llm_results"])
assert v1_schema_valid_count == 5
assert v1_validation_failure_count == 7

print(f"Query input: {len(query_inputs)}개, SHA256={query_input_hash}")
print(f"Allowed Accord: {len(allowed_accords)}개, SHA256={accord_vocabulary_hash}")
print(f"v1 운영 기준: schema-valid {v1_schema_valid_count}/12, validation failure {v1_validation_failure_count}/12")
display(pd.DataFrame(query_inputs)[["query_id", "resolution_type", "bridge_phrase"]])


Query input: 12개, SHA256=9171860b6f91178ed3a597afcc5d7ad5b903c0deb74a61c10d321782d8c3cb9b
Allowed Accord: 92개, SHA256=159bdc8fa6ce95d043efd369df042ffa4dd3d0b1a578ac57a9d9eae51b1daefc
v1 운영 기준: schema-valid 5/12, validation failure 7/12


,query_id,resolution_type,bridge_phrase
0,SQ0061,PURE,비 온 뒤의 숲의 냄새
1,SQ0032,PURE,숲 속에 온 듯한 향
2,SQ0136,PURE,빨래향 같이 자연스러운 향
3,SQ0105,PURE,따뜻한 햇살을 받으며 침대 위에 누워있는 듯한 느낌의 향수
4,SQ0012,PURE,샤워하고 나온듯한 따뜻하고 포근한 느낌의 향기
5,SQ0058,PURE,바다가 생각나는 시원한 향
6,SQ0002,MIXED,시원하고 깔끔한 향
7,SQ0047,MIXED,가벼우면서 시원한 느낌
8,SQ0051,MIXED,톡 쏘는 듯한 청량함과 방금 막 세탁된 코튼 향처럼 깨끗한 공기향
9,SQ0071,MIXED,겨울 아침에 딱~ 일어나서 창문을 열었는데 느껴지는 차갑고 상쾌한 향


## 2. Accord-only proposer와 validation

Prompt는 첫 호출 전에 아래 전문과 hash로 고정한다. 출력에는 `target_type`, rationale,
confidence, weight를 두지 않는다. Semantic 결과가 마음에 들지 않는다는 이유로 재호출하지 않고,
API 또는 JSON/schema validation 실패만 최대 2회 재시도한다.


In [3]:
SYSTEM_PROMPT = f'''You are the only proposer for Semantic Bridge Accord-only v2.
Read the Korean bridge_phrase exactly as supplied. Select only the Accord names that are most
strongly and defensibly connected to the phrase's actual scent characteristics.

You must return exactly one JSON object with exactly two keys: status and accords.

For a mapping, return this shape:
{{"status":"MAPPED","accords":["exact accord name"]}}

A second Accord is allowed only when it is independently strong and necessary:
{{"status":"MAPPED","accords":["exact accord name","exact accord name"]}}

When no Accord can be selected defensibly, return:
{{"status":"ABSTAIN","accords":[]}}

The allowed status values are listed separately:
- MAPPED
- ABSTAIN

Rules:
- Use only exact names from the closed Accord vocabulary below.
- MAPPED requires one or two unique Accord names.
- ABSTAIN requires an empty accords list.
- Do not choose an Accord merely because a surface word resembles its name.
- Do not force every emotion, temperature, texture, or scene into an Accord.
- Select only Accords supported by sufficiently clear scent characteristics in the phrase.
- Do not add a weak association merely to fill the second slot.
- Do not recommend perfumes.
- Do not output target_type, rationale, confidence, weight, aliases, or any extra key.
- Output JSON only, with no prose or code fence.

CLOSED ACCORD VOCABULARY ({len(allowed_accords)}):
{json.dumps(allowed_accords, ensure_ascii=False)}'''
USER_PROMPT_TEMPLATE = "bridge_phrase: {bridge_phrase}"
prompt_hash = json_hash({
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt_template": USER_PROMPT_TEMPLATE,
})


def parse_and_validate(content):
    try:
        text = str(content).strip()
        if text.startswith("```"):
            text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        payload = json.loads(text)
        if not isinstance(payload, dict):
            raise ValueError("top-level JSON object가 아닙니다.")
        if set(payload) != {"status", "accords"}:
            raise ValueError(f"schema key exact-match 실패: {sorted(payload)}")
        status, accords = payload["status"], payload["accords"]
        if status not in {"MAPPED", "ABSTAIN"}:
            raise ValueError(f"status 허용값 실패: {status!r}")
        if not isinstance(accords, list) or any(not isinstance(item, str) for item in accords):
            raise ValueError("accords는 string list여야 합니다.")
        if status == "MAPPED" and not 1 <= len(accords) <= 2:
            raise ValueError(f"MAPPED Accord 수 실패: {len(accords)}")
        if status == "ABSTAIN" and accords:
            raise ValueError("ABSTAIN accords가 비어 있지 않습니다.")
        unknown = [accord for accord in accords if accord not in allowed_accord_set]
        if unknown:
            raise ValueError(f"Accord vocabulary exact-match 실패: {unknown}")
        if len(accords) != len(set(accords)):
            raise ValueError("duplicate Accord")
        return {
            "validation_status": "VALID",
            "status": status,
            "accords": accords,
            "validation_error": "",
        }
    except Exception as error:
        return {
            "validation_status": "INVALID",
            "status": "",
            "accords": [],
            "validation_error": str(error),
        }


assert parse_and_validate('{"status":"MAPPED","accords":["' + allowed_accords[0] + '"]}')["validation_status"] == "VALID"
assert parse_and_validate('{"status":"ABSTAIN","accords":[]}')["validation_status"] == "VALID"
assert parse_and_validate('{"status":"MAPPED","accords":["__NOT_ALLOWED__"]}')["validation_status"] == "INVALID"
print(f"Prompt SHA256: {prompt_hash}")
print("Validation self-test: PASS")


Prompt SHA256: 38259633388dd0b990784876a5a18006023c3a31158b6a9570bbc6e30d8d6896
Validation self-test: PASS


In [4]:
def initial_checkpoint():
    return {
        "pilot_version": PILOT_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_v1": {
            **v1_input_hashes,
            "selection_annotation_sha256": v1_checkpoint["selection_annotation_sha256"],
            "schema_valid_count": v1_schema_valid_count,
            "validation_failure_count": v1_validation_failure_count,
        },
        "query_input_sha256": query_input_hash,
        "query_inputs": [
            {"query_id": row["query_id"], "bridge_phrase": row["bridge_phrase"]}
            for row in query_inputs
        ],
        "allowed_accords": {
            "source_path": str(ACCORD_DICTIONARY_PATH.relative_to(ROOT)),
            "source_sha256": file_sha256(ACCORD_DICTIONARY_PATH),
            "count": len(allowed_accords),
            "sha256": accord_vocabulary_hash,
            "items": allowed_accords,
        },
        "llm": {
            "model": MODEL_ID,
            "endpoint": ENDPOINT,
            "temperature": TEMPERATURE,
            "max_retries_after_initial": MAX_RETRIES,
            "timeout_seconds": REQUEST_TIMEOUT_SECONDS,
            "system_prompt": SYSTEM_PROMPT,
            "user_prompt_template": USER_PROMPT_TEMPLATE,
            "prompt_sha256": prompt_hash,
        },
        "output_schema": {
            "exact_keys": ["status", "accords"],
            "status_values": ["MAPPED", "ABSTAIN"],
            "mapped_accord_count": [1, 2],
            "abstain_accord_count": 0,
            "exact_vocabulary_match": True,
            "duplicates_allowed": False,
            "automatic_correction": False,
        },
        "results": [],
    }


def load_checkpoint():
    expected = initial_checkpoint()
    if not V2_CHECKPOINT_PATH.exists():
        write_json(V2_CHECKPOINT_PATH, expected)
        return expected
    checkpoint = json.loads(V2_CHECKPOINT_PATH.read_text(encoding="utf-8"))
    for key in ["pilot_version", "source_v1", "query_input_sha256", "query_inputs", "allowed_accords", "llm", "output_schema"]:
        if checkpoint.get(key) != expected.get(key):
            raise RuntimeError(f"기존 v2 checkpoint의 {key}가 현재 고정 조건과 다릅니다.")
    return checkpoint


def call_proposer(bridge_phrase, api_key):
    import requests

    response = requests.post(
        ENDPOINT,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={
            "model": MODEL_ID,
            "temperature": TEMPERATURE,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT_TEMPLATE.format(bridge_phrase=bridge_phrase)},
            ],
        },
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    payload = response.json()
    return payload, payload["choices"][0]["message"]["content"]


try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
api_key = os.getenv("GMS_KEY", "")
if not api_key:
    raise RuntimeError("GMS_KEY가 없어 v2를 실행할 수 없습니다.")

checkpoint = load_checkpoint()
completed_by_id = {
    row["query_id"]: row for row in checkpoint["results"] if row.get("completed")
}

for query in query_inputs:
    query_id = query["query_id"]
    if query_id in completed_by_id:
        continue
    attempts = []
    final_record = None
    for attempt_index in range(MAX_RETRIES + 1):
        try:
            raw_response, content = call_proposer(query["bridge_phrase"], api_key)
            validation = parse_and_validate(content)
            attempts.append({
                "attempt": attempt_index + 1,
                "called_at_utc": datetime.now(timezone.utc).isoformat(),
                "raw_response": raw_response,
                "content": content,
                "validation_error": validation["validation_error"],
            })
            if validation["validation_status"] == "VALID":
                final_record = {
                    "query_id": query_id,
                    "bridge_phrase": query["bridge_phrase"],
                    "completed": True,
                    "attempts": attempts,
                    **validation,
                }
                break
        except Exception as error:
            attempts.append({
                "attempt": attempt_index + 1,
                "called_at_utc": datetime.now(timezone.utc).isoformat(),
                "api_error": repr(error),
            })
        if attempt_index < MAX_RETRIES:
            time.sleep(min(2 ** attempt_index, 4))
    if final_record is None:
        if attempts and "content" in attempts[-1]:
            validation = parse_and_validate(attempts[-1]["content"])
            final_record = {
                "query_id": query_id,
                "bridge_phrase": query["bridge_phrase"],
                "completed": True,
                "attempts": attempts,
                **validation,
            }
        else:
            final_record = {
                "query_id": query_id,
                "bridge_phrase": query["bridge_phrase"],
                "completed": False,
                "attempts": attempts,
                "validation_status": "NOT_RUN",
                "status": "",
                "accords": [],
                "validation_error": "API_ERROR_AFTER_RETRIES",
            }
    checkpoint["results"] = [
        row for row in checkpoint["results"] if row["query_id"] != query_id
    ] + [final_record]
    write_json(V2_CHECKPOINT_PATH, checkpoint)

all_completed = (
    len(checkpoint["results"]) == 12
    and len({row["query_id"] for row in checkpoint["results"]}) == 12
    and all(row["completed"] for row in checkpoint["results"])
)
print(f"Completed LLM results: {sum(row['completed'] for row in checkpoint['results'])}/12")
if not all_completed:
    print("API error result는 completed=false로 보존했습니다. 동일 checkpoint에서만 재개합니다.")


Completed LLM results: 12/12


## 3. 최소 측정과 v1/v2 비교

자동 평가는 schema/vocabulary validation만 수행한다. Semantic 품질 정답이나 사람 점수는 만들지 않는다.
v1 raw 후보는 final raw response에서 표시용으로 추출할 뿐, v2 target으로 복사하거나 보정하지 않는다.


In [5]:
def raw_v1_candidates(v1_result):
    try:
        content = v1_result["attempts"][-1]["content"].strip()
        payload = json.loads(content)
        return [
            target.get("target_feature", "")
            for target in payload.get("targets", [])
            if isinstance(target, dict) and target.get("target_feature")
        ]
    except Exception:
        return []


comparison_df = pd.DataFrame()
if all_completed:
    result_by_id = {row["query_id"]: row for row in checkpoint["results"]}
    comparison_rows = []
    for query in query_inputs:
        query_id = query["query_id"]
        v1_result = v1_result_by_id[query_id]
        v2_result = result_by_id[query_id]
        comparison_rows.append({
            "query_id": query_id,
            "resolution_type": query["resolution_type"],
            "query_text": query["query_text"],
            "bridge_phrase": query["bridge_phrase"],
            "v1_raw_status": v1_result["raw_bridge_status"],
            "v1_raw_candidates_json": json.dumps(raw_v1_candidates(v1_result), ensure_ascii=False),
            "v1_validated_status": v1_result["validated_bridge_status"],
            "v1_validation_error": v1_result["validation_error"],
            "v2_validation_status": v2_result["validation_status"],
            "v2_status": v2_result["status"],
            "v2_accords_json": json.dumps(v2_result["accords"], ensure_ascii=False),
            "v2_accord_count": len(v2_result["accords"]),
            "v2_validation_error": v2_result["validation_error"],
            "v2_attempt_count": len(v2_result["attempts"]),
        })
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(COMPARISON_PATH, index=False, encoding="utf-8-sig")

    valid_output_count = int(comparison_df["v2_validation_status"].eq("VALID").sum())
    mapped_count = int(comparison_df["v2_status"].eq("MAPPED").sum())
    abstain_count = int(comparison_df["v2_status"].eq("ABSTAIN").sum())
    validation_failure_count = int(comparison_df["v2_validation_status"].eq("INVALID").sum())
    accord_count_distribution = {
        str(key): int(value)
        for key, value in comparison_df.loc[
            comparison_df["v2_validation_status"].eq("VALID"), "v2_accord_count"
        ].value_counts().sort_index().items()
    }
    validation_failure_reasons = comparison_df.loc[
        comparison_df["v2_validation_status"].eq("INVALID"), "v2_validation_error"
    ].value_counts().to_dict()
    checkpoint["summary"] = {
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "valid_output_count": valid_output_count,
        "mapped_count": mapped_count,
        "abstain_count": abstain_count,
        "accord_count_distribution": accord_count_distribution,
        "validation_failure_count": validation_failure_count,
        "validation_failure_reasons": validation_failure_reasons,
        "comparison_path": str(COMPARISON_PATH.relative_to(ROOT)),
        "comparison_sha256": file_sha256(COMPARISON_PATH),
        "retrieval_executed": False,
        "human_evaluation_created": False,
    }
    write_json(V2_CHECKPOINT_PATH, checkpoint)

    display(pd.DataFrame([checkpoint["summary"]]).drop(columns=["validation_failure_reasons"]))
    display(comparison_df[[
        "query_id", "bridge_phrase", "v1_raw_candidates_json",
        "v1_validation_error", "v2_validation_status", "v2_status",
        "v2_accords_json", "v2_validation_error",
    ]])
    print(f"Comparison: {COMPARISON_PATH}")


,completed_at_utc,valid_output_count,mapped_count,abstain_count,accord_count_distribution,validation_failure_count,comparison_path,comparison_sha256,retrieval_executed,human_evaluation_created
0,2026-09-02T06:06:44.573558+00:00,12,10,2,"{'0': 2, '1': 2, '2': 8}",0,analysis_outputs\23_semantic_bridge_accord_onl...,e09189d2b02a7dcb53a3c3636df4d8857dc8301d71d530...,False,False


,query_id,bridge_phrase,v1_raw_candidates_json,v1_validation_error,v2_validation_status,v2_status,v2_accords_json,v2_validation_error
0,SQ0061,비 온 뒤의 숲의 냄새,"[""foresty"", ""wet plaster"", ""earthy""]",target_type 오류: 'ACCORD|CANONICAL_NOTE',VALID,MAPPED,"[""foresty"", ""earthy""]",
1,SQ0032,숲 속에 온 듯한 향,"[""foresty"", ""woody""]",target_type 오류: 'ACCORD|CANONICAL_NOTE',VALID,MAPPED,"[""foresty""]",
2,SQ0136,빨래향 같이 자연스러운 향,"[""soapy"", ""fresh""]",target_type 오류: 'ACCORD|CANONICAL_NOTE',VALID,MAPPED,"[""fresh"", ""soapy""]",
3,SQ0105,따뜻한 햇살을 받으며 침대 위에 누워있는 듯한 느낌의 향수,[],,VALID,ABSTAIN,[],
4,SQ0012,샤워하고 나온듯한 따뜻하고 포근한 느낌의 향기,"[""soapy"", ""warm spicy"", ""sweet""]",,VALID,MAPPED,"[""soapy"", ""warm spicy""]",
5,SQ0058,바다가 생각나는 시원한 향,"[""aquatic"", ""Marine"", ""fresh""]",target_type 오류: 'ACCORD|CANONICAL_NOTE',VALID,MAPPED,"[""aquatic"", ""fresh""]",
6,SQ0002,시원하고 깔끔한 향,"[""fresh"", ""soapy""]",target_type 오류: 'ACCORD|CANONICAL_NOTE',VALID,MAPPED,"[""fresh""]",
7,SQ0047,가벼우면서 시원한 느낌,"[""fresh"", ""aquatic""]",,VALID,MAPPED,"[""fresh"", ""citrus""]",
8,SQ0051,톡 쏘는 듯한 청량함과 방금 막 세탁된 코튼 향처럼 깨끗한 공기향,"[""Citrus"", ""Fresh"", ""Powdery""]",target_type 오류: 'ACCORD|CANONICAL_NOTE',VALID,MAPPED,"[""fresh"", ""soapy""]",
9,SQ0071,겨울 아침에 딱~ 일어나서 창문을 열었는데 느껴지는 차갑고 상쾌한 향,"[""fresh"", ""citrus"", ""ozonic""]",,VALID,MAPPED,"[""fresh"", ""citrus""]",


Comparison: C:\Users\SSAFY\Desktop\EDA\analysis_outputs\23_semantic_bridge_accord_only_v2_comparison.csv


In [6]:
if not all_completed:
    CURRENT_STATE = "WAITING_FOR_API_RECOVERY"
else:
    CURRENT_STATE = "COMPLETE"

summary = checkpoint.get("summary", {})
summary_lines = [
    "## 현재 상태: `" + CURRENT_STATE + "`",
    f"- Valid output: {summary.get('valid_output_count', 'PENDING')}/12",
    f"- MAPPED: {summary.get('mapped_count', 'PENDING')}",
    f"- ABSTAIN: {summary.get('abstain_count', 'PENDING')}",
    f"- Validation failure: {summary.get('validation_failure_count', 'PENDING')}",
    "- Retrieval 실행: 없음",
    "- 사람 평가 파일 생성: 없음",
    "- Canonical Note 사용: 없음",
]
display(Markdown("\n\n".join(summary_lines)))


## 현재 상태: `COMPLETE`

- Valid output: 12/12

- MAPPED: 10

- ABSTAIN: 2

- Validation failure: 0

- Retrieval 실행: 없음

- 사람 평가 파일 생성: 없음

- Canonical Note 사용: 없음